## Try find Cluster using Unsupervised algorithms


In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import RobustScaler
from category_encoders import TargetEncoder

from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

import umap
import hdbscan

# =====================================================
# LOAD DATA
# =====================================================

df1 = pd.read_csv('HealthCareAnalytics.csv')
df = df1.copy()
# =====================================================
# BASIC CLEANING
# =====================================================

df = df.drop_duplicates()

target = 'Stay'

# encode target
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df[target] = le.fit_transform(df[target])

# =====================================================
# SPLIT FEATURES
# =====================================================

X = df.drop(columns=[target])
y = df[target]

# =====================================================
# HANDLE MISSING VALUES
# =====================================================

for col in X.columns:
    
    if X[col].dtype == 'object':
        X[col] = X[col].fillna(X[col].mode()[0])
        
    else:
        X[col] = X[col].fillna(X[col].median())

# =====================================================
# CATEGORICAL + NUMERICAL
# =====================================================

cat_cols = X.select_dtypes(include='object').columns
num_cols = X.select_dtypes(exclude='object').columns

# =====================================================
# TARGET ENCODING
# =====================================================

te = TargetEncoder()

X[cat_cols] = te.fit_transform(X[cat_cols], y)

# =====================================================
# SCALING
# =====================================================

scaler = RobustScaler()

X_scaled = scaler.fit_transform(X)

# =====================================================
# UMAP REDUCTION
# =====================================================

reducer = umap.UMAP(
    n_components=10,
    n_neighbors=30,
    min_dist=0.1,
    metric='euclidean',
    random_state=42
)

X_umap = reducer.fit_transform(X_scaled)

print(X_umap.shape)

# =====================================================
# HDBSCAN CLUSTERING
# =====================================================

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=500,
    min_samples=30,
    metric='euclidean',
    cluster_selection_method='eom'
)

clusters = clusterer.fit_predict(X_umap)

# =====================================================
# NUMBER OF CLUSTERS
# =====================================================

n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)

print(f'Clusters Found: {n_clusters}')

# =====================================================
# REMOVE NOISE POINTS
# =====================================================

mask = clusters != -1

X_valid = X_umap[mask]
clusters_valid = clusters[mask]

# =====================================================
# CLUSTER QUALITY
# =====================================================

sil = silhouette_score(X_valid, clusters_valid)

db = davies_bouldin_score(X_valid, clusters_valid)

ch = calinski_harabasz_score(X_valid, clusters_valid)

print(f'Silhouette Score : {sil:.4f}')
print(f'Davies Bouldin : {db:.4f}')
print(f'Calinski Score : {ch:.4f}')

# =====================================================
# CLUSTER DISTRIBUTION
# =====================================================

cluster_counts = pd.Series(clusters).value_counts()

print(cluster_counts)

# =====================================================
# ATTACH CLUSTERS
# =====================================================

df['cluster'] = clusters

# =====================================================
# ANALYZE CLUSTERS
# =====================================================

cluster_summary = df.groupby('cluster').mean(numeric_only=True)

print(cluster_summary)

C:\Users\kadam\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


(318438, 10)
Clusters Found: 153
Silhouette Score : 0.6453
Davies Bouldin : 0.6217
Calinski Score : 688012.1377
-1      22659
 9      18039
 3      17935
 17      9998
 10      9515
        ...  
 58       529
 96       517
 124      507
 108      507
 97       504
Name: count, Length: 154, dtype: int64
               case_id  Hospital_code  City_Code_Hospital  \
cluster                                                     
-1       165058.611722      18.209497            4.623770   
 0       178167.359494      21.329114            7.000000   
 1       184999.222469      22.137931            2.007786   
 2       190647.843112      13.515306            4.051020   
 3       154664.311068      13.376861            4.052133   
...                ...            ...                 ...   
 148     166531.876772      15.411123            1.003272   
 149     166836.542354      24.300330            5.566557   
 150     142861.968158      23.191050            5.936317   
 151     146806.492665  

In [9]:
df['Stay'].value_counts()

Stay
1     87491
10    78139
2     55159
4     35018
0     23604
3     11743
6     10254
9      6683
7      4838
8      2765
5      2744
Name: count, dtype: int64

In [10]:
df['cluster'].value_counts()

cluster
-1      22659
 9      18039
 3      17935
 17      9998
 10      9515
        ...  
 58       529
 96       517
 124      507
 108      507
 97       504
Name: count, Length: 154, dtype: int64

In [16]:
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

# =====================================================
# KMEANS (K = 11)
# =====================================================

k = 11

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(X_umap)

# =====================================================
# EVALUATION
# =====================================================

sil = silhouette_score(
    X_umap,
    clusters
)

db = davies_bouldin_score(
    X_umap,
    clusters
)

ch = calinski_harabasz_score(
    X_umap,
    clusters
)

print(f"K = {k}")
print(f"Silhouette Score: {sil:.4f}")
print(f"Davies-Bouldin: {db:.4f}")
print(f"Calinski-Harabasz: {ch:.4f}")

# =====================================================
# CLUSTER DISTRIBUTION
# =====================================================

cluster_counts = pd.Series(
    clusters
).value_counts().sort_index()

print(cluster_counts)

# =====================================================
# ATTACH CLUSTER
# =====================================================

df['cluster'] = clusters

K = 11
Silhouette Score: 0.4259
Davies-Bouldin: 0.9774
Calinski-Harabasz: 107512.7978
0     55067
1     40049
2     27474
3     19719
4     36923
5     18067
6     19576
7     11633
8     10316
9      9515
10    70099
Name: count, dtype: int64


In [ ]:
df1['Stay'].value_counts()

**OBSERVATIONS : I removed the target variable and applied DBSCAN clustering to identify natural patterns in the dataset using the Silhouette Score for evaluation. However, the generated clusters were significantly different from the actual target classes, and the true class labels showed a much lower Silhouette Score. This indicates that the dataset is highly noisy and lacks well-separated intrinsic cluster structures aligned with the target variable.**